# Lesson 07 — What if gravity were a different theory?

**Phase 5 of the gravsim project, part II: the PPN parameters $\beta$ and $\gamma$.**

General Relativity passed Mercury's test in 1915. But a test is only as good as the
alternatives it excludes, and by the 1960s there were many: scalar–tensor theories
(Brans–Dicke), vector–tensor theories, theories with preferred frames. Nordtvedt and Will
built a common language for all of them, the *parametrised post-Newtonian* (PPN) framework:
in the weak-field, slow-motion limit every metric theory of gravity is described by a handful
of numbers, and each solar-system experiment measures one combination of them. This lesson
turns gravsim into a PPN laboratory and asks what Mercury's perihelion, by itself, can and
cannot decide.

By the end you should be able to

1. define $\gamma$ and $\beta$ and give their values in GR and in Brans–Dicke theory;
2. derive the factor $(2+2\gamma-\beta)/3$ in the perihelion advance;
3. measure that factor on a grid of theories and show that $\gamma$ counts twice as much as
   $\beta$;
4. place the Cassini and lunar-laser-ranging bounds on the $(\beta,\gamma)$ plane and explain
   why Mercury alone leaves a degeneracy that only a light-propagation experiment breaks.

## 1. The physics: two numbers for any metric theory

**The PPN metric.** Around a static mass, in isotropic coordinates and with $U = GM/(c^2 r)$,
every metric theory with a Newtonian limit has, to the order that matters in the solar system,

$$
g_{00} = -\big(1 - 2U + 2\beta U^2\big), \qquad g_{ij} = \big(1 + 2\gamma U\big)\,\delta_{ij} .
$$

The $-2U$ term is Newton's; it is the same in every theory, because that is what "Newtonian
limit" means. The two numbers that follow are the theory:

* $\gamma$ — **how much space curvature a unit of mass produces.** It governs everything light
  does (deflection $\propto(1+\gamma)/2$, Shapiro delay $\propto(1+\gamma)$) and the
  velocity-dependent part of the force on a planet.
* $\beta$ — **how non-linear the superposition of gravity is**, the coefficient of the $U^2$
  term in $g_{00}$: the field of two masses is not quite the sum of their fields, and
  gravitational energy itself gravitates.

GR has $\beta=\gamma=1$. Brans–Dicke theory, with its coupling constant $\omega$, has
$\gamma = (1+\omega)/(2+\omega)$, $\beta = 1$: the scalar field dilutes the space curvature.
A theory with $\gamma = 0$ would have Newtonian time dilation but flat space; the
Nordtvedt parameter $\eta = 4\beta-\gamma-3$ measures the violation of the strong
equivalence principle (do bodies with different gravitational binding energy fall alike?).

**The equations of motion.** Expanding the geodesic equation of this metric gives the
test-body acceleration of Lesson 04 with $\beta,\gamma$ in place of $1,1$,

$$
\mathbf a = -\frac{GM}{r^3}\mathbf r + \frac{GM}{c^2r^3}\Big[\Big(2(\beta+\gamma)\frac{GM}{r} - \gamma v^2\Big)\mathbf r + 2(1+\gamma)(\mathbf r\cdot\mathbf v)\,\mathbf v\Big],
$$

and the full $N$-body (EIH) equations with the PPN coefficients listed in Lesson 06
(`make_eih(c, beta, gamma)`; `make_pn1` takes the same two knobs).

**The perihelion factor.** Write the correction in the orbital plane as a radial component
$R$ and a transverse one $T$. With $k = GM/p$, $p = a(1-e^2)$ and the Kepler relations
$GM/r = k(1+e\cos\nu)$, $v^2 = k(1+2e\cos\nu+e^2)$, $\dot r^2 = ke^2\sin^2\nu$,
$r\dot\nu = \sqrt k(1+e\cos\nu)$, Gauss's planetary equation for the perihelion,
$d\omega/d\nu = (r^2/GMe)\,[-R\cos\nu + T(1+r/p)\sin\nu]$, becomes

$$
\frac{d\omega}{d\nu} = \frac{k}{c^2e}\Big\{-\cos\nu\big[2(\beta+\gamma)(1+e\cos\nu) - \gamma(1+2e\cos\nu+e^2) + 2(1+\gamma)e^2\sin^2\nu\big] + 2(1+\gamma)\,e\sin^2\nu\,(2+e\cos\nu)\Big\}.
$$

Integrating over one revolution kills every odd power of $\cos\nu$ and leaves
$\pi e\,[-2(\beta+\gamma) + 2\gamma + 4(1+\gamma)] = \pi e\,(4+4\gamma-2\beta)$, so

$$
\boxed{\;\Delta\varpi = \frac{2+2\gamma-\beta}{3}\cdot\frac{6\pi GM}{c^2 a(1-e^2)}\;}
$$

(Robertson 1962; Weinberg 1972 §8.6; Will 2014 eq. 68; full derivation in
`docs/theory/ppn.md`). The split is instructive: $\gamma$ enters with weight $2$ — through
both the velocity-dependent force and the $GM/r$ term — and $\beta$ with weight $-1$: more
non-linearity *reduces* the advance. Mercury therefore measures the single combination
$2\gamma-\beta$, and nothing else.

**The bounds.** Cassini's 2003 solar-conjunction experiment measured the Shapiro delay to
$\gamma-1 = (2.1\pm2.3)\times10^{-5}$ (Bertotti, Iess & Tortora 2003). Lunar laser ranging
constrains the Nordtvedt effect, $\eta = 4\beta-\gamma-3 = (4.4\pm4.5)\times10^{-4}$
(Williams, Turyshev & Boggs 2004). Mercury's perihelion — radar ranging from the 1960s,
MESSENGER from 2011 to 2015 — fixes $2\gamma-\beta-1$ to about $10^{-4}$ once the solar
$J_2$ is taken from helioseismology (Will 2014, Table 4; Park et al. 2017). The PPN expansion
inherits the validity limit of 1PN: $v/c\ll1$, which the $c/100$ knob respects.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import (
    gr_precession_arcsec_per_century,
    perihelion_rate_difference,
    state_to_elements,
)
from gravsim.data.horizons import load_j2000
from gravsim.forces import accel_newton, make_eih
from gravsim.integrators import integrate_scipy
from gravsim.viz import PALETTE, figure_grid

plt.rcParams["figure.dpi"] = 100


def ppn_factor(beta, gamma):
    """The PPN perihelion factor (2 + 2 gamma - beta) / 3; 1 in General Relativity."""
    return (2.0 + 2.0 * gamma - beta) / 3.0


two = load_j2000().subset(("Sun", "Mercury")).to_barycentric()
r0, v0 = two.relative_to("Sun")
gm = float(two.gm.sum())
el = state_to_elements(r0[1], v0[1], gm)
a, e, period = (float(el[k]) for k in ("a", "e", "period"))

## 2. The factor across the plane

Before running anything: the factor $(2+2\gamma-\beta)/3$ over the $(\beta,\gamma)$ plane, with
a few theories marked. The white line is $2\gamma-\beta=1$: every theory on it gives Mercury
exactly GR's advance. Brans–Dicke with $\omega=5$ sits at $\gamma=6/7$ and would have given
Einstein $90\%$ of the answer; a theory with no space curvature, one third of it; a theory with
$\beta=2$, $\gamma=0$ would have left Le Verrier nothing to explain.

In [ ]:
betas = np.linspace(0.0, 2.0, 201)
gammas = np.linspace(0.0, 2.0, 201)
B, G = np.meshgrid(betas, gammas)
F = ppn_factor(B, G)
theories = {
    "GR (β = γ = 1)": (1.0, 1.0),
    "Brans–Dicke, ω = 5 (γ = 6/7)": (1.0, 6.0 / 7.0),
    "no space curvature (γ = 0)": (1.0, 0.0),
    "linear superposition (β = 0)": (0.0, 1.0),
    "no advance at all (β = 2, γ = 0)": (2.0, 0.0),
}
fig, axes = figure_grid(1, panel=(7.0, 5.4))
ax = axes[0]
cs = ax.contourf(B, G, F, levels=np.linspace(0.0, 2.0, 21), cmap="viridis")
fig.colorbar(cs, ax=ax, label="(2 + 2γ − β) / 3")
ax.contour(B, G, F, levels=[1.0], colors="w", linewidths=1.5)
for k, (name, (b, g)) in enumerate(theories.items()):
    ax.plot(b, g, "o", color=PALETTE[k], mec="w", ms=9, label=f"{name}: ×{ppn_factor(b, g):.3f}")
ax.set_xlabel("β  (non-linearity of superposition)")
ax.set_ylabel("γ  (space curvature per unit mass)")
ax.set_title("Mercury's advance relative to GR;  white line: 2γ − β = 1")
ax.legend(fontsize=7, loc="upper left")

## 3. The experiment: a grid of theories

Sun + Mercury with the EIH law at $c/100$ (the runs take a second each and the advance is
$0.29°$ per orbit, Lesson 04 §6), 30 orbits, one Newtonian run on the same grid for the
differencing. Nine theories, $\beta,\gamma\in\{0,1,2\}$. The prediction is the factor times the
GR rate for the same $c$; the measured rates land on it to better than $1\%$. The residual
grows with $\beta$ and $\gamma$ because it is second order in $\mu/p$ (Lesson 04 §6) with
coefficients that depend on the theory — the price of the $c/100$ knob, not a failure of the
formula. It holds for the theory with no advance at all, where the fit returns $-0.2\%$ of GR,
and for $\beta=0,\gamma=2$, where it returns twice GR.

In [ ]:
c_small = C.C_AU_DAY / 100.0
t_eval = two.t + np.arange(0.0, 30 * period, 0.25)
newton = integrate_scipy(two, accel_newton, t_eval)
gr_rate = float(gr_precession_arcsec_per_century(a, e, gm, c=c_small))
grid = (0.0, 1.0, 2.0)
measured = {}
print(f"{'beta':>5s} {'gamma':>6s} {'factor':>7s} {'predicted':>10s} {'measured':>10s} {'± err':>6s}   [arcsec/cy at c/100]")
for beta in grid:
    for gamma in grid:
        run = integrate_scipy(two, make_eih(c=c_small, beta=beta, gamma=gamma), t_eval)
        rate, err = perihelion_rate_difference(run, newton, "Mercury")
        measured[(beta, gamma)] = (rate, err)
        print(f"{beta:5.1f} {gamma:6.1f} {ppn_factor(beta, gamma):7.3f} {ppn_factor(beta, gamma) * gr_rate:10.0f} {rate:10.0f} {err:6.0f}")
print(f"\nGR rate at c/100: {gr_rate:.0f} arcsec/cy = {gr_rate / (C.JULIAN_CENTURY_DAYS / period) / 3600:.3f} deg/orbit")

In [ ]:
fig, axes = figure_grid(2, panel=(5.8, 4.3))
ax = axes[0]
for (b, g), (rate, err) in measured.items():
    ax.errorbar(ppn_factor(b, g), rate / gr_rate, yerr=err / gr_rate, fmt="o", color=PALETTE[0], ms=5)
    ax.annotate(f"({b:g}, {g:g})", (ppn_factor(b, g), rate / gr_rate), fontsize=7,
                xytext=(5, 4) if b == 2.0 else (5, -9), textcoords="offset points")
lim = np.array([-0.15, 2.15])
ax.plot(lim, lim, "-", color="0.3", lw=1, label="measured = (2 + 2γ − β)/3")
ax.set_xlabel("predicted factor (2 + 2γ − β)/3")
ax.set_ylabel("measured rate / GR rate")
ax.grid(True, lw=0.3); ax.legend(fontsize=8, loc="upper left")
ax.set_title("nine theories (β, γ), EIH runs at c/100")

ax = axes[1]
xs = np.linspace(0.0, 2.0, 50)
ax.plot(xs, ppn_factor(1.0, xs), "-", color=PALETTE[1], lw=1, label="γ varies at β = 1: slope +2/3")
ax.plot(xs, ppn_factor(xs, 1.0), "-", color=PALETTE[2], lw=1, label="β varies at γ = 1: slope −1/3")
ax.plot(grid, [measured[(1.0, g)][0] / gr_rate for g in grid], "o", color=PALETTE[1], ms=6)
ax.plot(grid, [measured[(b, 1.0)][0] / gr_rate for b in grid], "s", color=PALETTE[2], ms=6)
ax.set_xlabel("β or γ"); ax.set_ylabel("measured rate / GR rate")
ax.grid(True, lw=0.3); ax.legend(fontsize=8)
ax.set_title("space curvature counts twice, non-linearity once and negatively")

## 4. A theory Mercury cannot tell from GR

Take $\beta = 1.4$, $\gamma = 1.2$: forty percent more non-linearity, twenty percent more space
curvature, and $2\gamma-\beta = 1$. Mercury's perihelion advances exactly as in GR — the run
confirms it to $0.2\%$, the second-order remainder again. Yet this theory is wildly wrong
elsewhere: light passing the Sun
would be deflected by $(1+\gamma)/2 = 1.1$ times Einstein's $1.75''$, the Shapiro delay would be
$10\%$ longer, and $\eta = 4\beta-\gamma-3 = 1.4$ would make the Earth and the Moon fall toward
the Sun at measurably different rates. A perihelion measures one combination; it takes an
experiment with *light* to measure $\gamma$ on its own.

In [ ]:
fake = (1.4, 1.2)  # 2 gamma - beta = 1: the same perihelion advance as GR
run_fake = integrate_scipy(two, make_eih(c=c_small, beta=fake[0], gamma=fake[1]), t_eval)
rate_fake, err_fake = perihelion_rate_difference(run_fake, newton, "Mercury")
rate_gr, err_gr = measured[(1.0, 1.0)]
print(f"(β, γ) = {fake}: factor {ppn_factor(*fake):.3f}")
print(f"Mercury at c/100:  GR {rate_gr:.0f} ± {err_gr:.0f},  this theory {rate_fake:.0f} ± {err_fake:.0f} arcsec/cy   (ratio {rate_fake / rate_gr:.4f})")
print(f"light deflection at the limb: GR {C.LIGHT_DEFLECTION_LIMB_ARCSEC:.3f} arcsec, this theory "
      f"{(1 + fake[1]) / 2 * C.LIGHT_DEFLECTION_LIMB_ARCSEC:.3f} arcsec   (factor (1 + γ)/2 = {(1 + fake[1]) / 2:.2f})")
print(f"Nordtvedt parameter η = 4β − γ − 3:  GR 0,  this theory {4 * fake[0] - fake[1] - 3:.2f}")

## 5. The bounds, to scale

Each experiment is a band in the $(\beta,\gamma)$ plane: Cassini is a horizontal strip
($\gamma$ alone), lunar laser ranging a steep one ($4\beta-\gamma$), Mercury a strip of slope
$1/2$ ($2\gamma-\beta$). On the scale of the plot — $10^{-4}$ per division — the three cross at
General Relativity, and each pair of them pins both parameters. Mercury and Cassini together
give $\beta-1 = 2(\gamma-1) - (2\gamma-\beta-1)$ with an uncertainty of $\sim10^{-4}$.

What limits Mercury's own band? Not the measurement of the advance: our simulation already
reproduces the differenced rate to $10^{-3}$ in ten years and to $2\times10^{-5}$ in a century
(Lesson 04, `scripts/validate.py`), and MESSENGER ranging did comparably well. The limit is the
*model*: the solar quadrupole $J_2$ also advances the perihelion ($0.03''$/cy, $7\times10^{-4}$ of
the GR term) and, from Mercury alone, is degenerate with $2\gamma-\beta$; helioseismology
($J_2 = (2.2\pm0.1)\times10^{-7}$) is what breaks that degeneracy, as Cassini breaks the other.

In [ ]:
bounds = {  # A (beta - 1) + B (gamma - 1) = value ± sigma
    "Cassini 2003, Shapiro delay:  γ − 1": (0.0, 1.0, 2.1e-5, 2.3e-5),
    "Lunar laser ranging:  η = 4β − γ − 3": (4.0, -1.0, 4.4e-4, 4.5e-4),
    "Mercury's perihelion:  2γ − β − 1": (-1.0, 2.0, 0.0, 8.0e-5),
}
scale, ext = 1e4, 6.0
x = np.linspace(-ext, ext, 2)
fig, axes = figure_grid(1, panel=(6.8, 5.6))
ax = axes[0]
for k, (name, (A, Bc, val, sig)) in enumerate(bounds.items()):
    lo = ((val - sig) * scale - A * x) / Bc
    hi = ((val + sig) * scale - A * x) / Bc
    ax.fill_between(x, lo, hi, color=PALETTE[k], alpha=0.25, lw=0)
    ax.plot(x, (val * scale - A * x) / Bc, "-", color=PALETTE[k], lw=1.2, label=f"{name} = ({val * 1e4:+.2f} ± {sig * 1e4:.2f}) × 10⁻⁴")
ax.plot(0.0, 0.0, "*", color="k", ms=13, label="General Relativity")
ax.set_xlim(-ext, ext); ax.set_ylim(-ext, ext); ax.set_aspect("equal")
ax.set_xlabel("(β − 1) × 10⁴"); ax.set_ylabel("(γ − 1) × 10⁴")
ax.grid(True, lw=0.3); ax.legend(fontsize=7, loc="lower right")
ax.set_title("solar-system bounds on the PPN parameters")

sig_cassini, sig_mercury = bounds["Cassini 2003, Shapiro delay:  γ − 1"][3], bounds["Mercury's perihelion:  2γ − β − 1"][3]
print(f"Mercury + Cassini:  σ(β − 1) = sqrt((2 σ_γ)² + σ_Mercury²) = {math.hypot(2 * sig_cassini, sig_mercury):.1e}")
print(f"Mercury alone, with J2 unknown: the 0.03 arcsec/cy of the solar quadrupole is {0.03 / C.MERCURY_PRECESSION_GR_ARCSEC_CY:.1e} of the GR term, "
      f"i.e. 2γ − β − 1 to ±{3 * 0.03 / C.MERCURY_PRECESSION_GR_ARCSEC_CY:.1e}")
print(f"our own measurement noise, 10 yr differenced run: 0.028 / 42.98 → 2γ − β − 1 to ±{3 * 0.028 / C.MERCURY_PRECESSION_GR_ARCSEC_CY:.1e}; "
      f"century run: 0.001 / 42.98 → ±{3 * 0.001 / C.MERCURY_PRECESSION_GR_ARCSEC_CY:.1e}")

## 6. Brans–Dicke at the real $c$, and Le Verrier's 38 arcseconds

One alternative at the real speed of light, ten years, EIH: Brans–Dicke with $\omega=5$,
$\gamma = 6/7$. The prediction is $0.905\times42.98 = 38.9''$/cy, and the differenced rate finds
it. The historical twist: Le Verrier's *original* residual was $38''$, not $43''$. Had the
alternative theories existed in 1859, a scalar–tensor theory with $\omega\approx4$ would have
fitted his number — and Newcomb's 1882 revision would have ruled it out. Cassini today requires
$\omega > 4\times10^4$.

In [ ]:
omega = 5.0
gamma_bd = (1.0 + omega) / (2.0 + omega)
t10 = two.t + np.arange(int(round(10.0 * C.JULIAN_YEAR_DAYS)) + 1, dtype=float)
newton10 = integrate_scipy(two, accel_newton, t10)
bd10 = integrate_scipy(two, make_eih(c=C.C_AU_DAY, beta=1.0, gamma=gamma_bd), t10)
rate_bd, err_bd = perihelion_rate_difference(bd10, newton10, "Mercury")
pred_bd = float(gr_precession_arcsec_per_century(a, e, gm, beta=1.0, gamma=gamma_bd))
print(f"Brans–Dicke ω = {omega:g}: γ = {gamma_bd:.4f}, factor {ppn_factor(1.0, gamma_bd):.4f}")
print(f"  predicted {pred_bd:.3f} arcsec/cy,  measured {rate_bd:.3f} ± {err_bd:.3f}  (GR: {C.MERCURY_PRECESSION_GR_ARCSEC_CY})")
f_lv = 38.0 / C.MERCURY_PRECESSION_GR_ARCSEC_CY
gamma_lv = (3.0 * f_lv - 1.0) / 2.0
omega_lv = (2.0 * gamma_lv - 1.0) / (1.0 - gamma_lv)
print(f"Le Verrier's 38 arcsec/cy = {f_lv:.3f} × GR  →  γ = {gamma_lv:.3f} (β = 1), Brans–Dicke ω = {omega_lv:.1f}")
print(f"Cassini (|γ − 1| < 2.3e-5 at 1σ) requires ω > {1 / 2.3e-5 - 2:.0f}")

## 7. Summary

* Every metric theory looks the same at Newtonian order; at the next order it is described by
  $\gamma$ (space curvature per unit mass) and $\beta$ (non-linearity). GR: $\beta=\gamma=1$;
  Brans–Dicke: $\gamma=(1+\omega)/(2+\omega)$.
* Gauss's equation averaged over a Kepler orbit gives
  $\Delta\varpi = \tfrac{2+2\gamma-\beta}{3}\,6\pi GM/(c^2a(1-e^2))$; the EIH runs reproduce the
  factor on a $3\times3$ grid of theories to better than $1\%$, including zero and twice GR.
* Mercury measures only $2\gamma-\beta$: a theory with $\beta=1.4,\gamma=1.2$ gives the same
  $43''$. Cassini ($\gamma$ from the Shapiro delay) and lunar laser ranging ($4\beta-\gamma$)
  break the degeneracy; the solar $J_2$ is the other one, broken by helioseismology.
* At the real $c$, Brans–Dicke $\omega=5$ gives $38.9''$/cy — Le Verrier's own $38''$ would have
  fitted $\omega\approx4$.

## Exercises

1. Repeat §3 with `make_newton_pn1` instead of `make_eih`. Are the measured factors the same?
   (They should be: for a test body the two laws are identical, Lesson 06 §1.)
2. Find the theory on the white line of §2 with $\gamma = 0.5$. Run it and confirm the GR rate;
   then compute what it predicts for light deflection.
3. Lesson 05's exact Schwarzschild orbit has no $\beta,\gamma$. Why can't the PPN playground be
   extended to the strong field with the same formulas?
4. ★ Measure the $J_2$ degeneracy: add the $J_2$ force of Lesson 03 with $J_2 = 2.2\times10^{-7}$
   to a GR run and find the $(\beta,\gamma)$ on the $\gamma = 1$ line that reproduces its rate
   without $J_2$. Then do the same for Venus: does the degeneracy survive a second planet?
5. ★ The Nordtvedt effect: in a theory with $\eta\neq0$ the Earth (binding energy
   $4.6\times10^{-10}$ of its mass) and the Moon ($2\times10^{-11}$) fall differently toward the Sun,
   polarising the lunar orbit by $\sim13\,\eta$ m. Estimate the $\eta$ that 1-cm laser ranging
   can detect and compare with the band of §5.